# Import de PySpark et des clés

In [2]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import numpy as np
from datetime import datetime, date

import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql import Row
spark = SparkSession.builder \
    .appName("BGES") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

In [3]:
import pandas as pd 
mat_inf = pd.read_csv('BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BERLIN_INFORMATIQUE/MATERIEL_INFORMATIQUE_20260429.txt',sep=';').columns
personnel = pd.read_csv('BDD_BGES/BDD_BGES_BERLIN/PERSONNEL_BERLIN.txt',sep=';').columns
mission = pd.read_csv('BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BERLIN_MISSION/MISSION_20260429.txt',sep=';').columns
print(pd.read_csv('BDD_BGES/BDD_BGES_BERLIN/PERSONNEL_BERLIN.txt',sep=';')['TS_CREATION_PERSONNEL'])
mat_inf = list(mat_inf)
personnel = list(personnel)
mission = list(mission)

KeyPersoMission = []
for i in personnel :
    if i in mission : 
        KeyPersoMission.append(i)

KeyPersoMission

0       2000-05-09 11:54:25
1       2014-08-17 01:27:04
2       2013-01-22 14:11:50
3       2007-08-27 01:28:18
4       2012-03-02 18:24:08
               ...         
4238    2002-07-10 13:02:23
4239    2023-06-25 19:12:37
4240    2020-06-27 04:11:52
4241    2007-06-06 12:15:27
4242    2003-10-02 11:51:43
Name: TS_CREATION_PERSONNEL, Length: 4243, dtype: object


['ID_PERSONNEL', 'NOM_PERSONNEL', 'PRENOM_PERSONNEL']

In [4]:
print(mat_inf)
print(personnel)
print(mission)

['ID_MATERIELINFO', 'ID_PERSONNEL', 'NOM_PERSONNEL', 'PRENOM_PERSONNEL', 'DATE_ACHAT', 'TYPE', 'MODELE']
['ID_PERSONNEL', 'NOM_PERSONNEL', 'PRENOM_PERSONNEL', 'DT_NAISS', 'VILLE_NAISS', 'PAYS_NAISS', 'NUM_SECU', 'IND_PAYS_NUM_TELP', 'NUM_TELEPHONE', 'NUM_VOIE', 'DSC_VOIE', 'CMPL_VOIE', 'CD_POSTAL', 'VILLE', 'PAYS', 'FONCTION_PERSONNEL', 'TS_CREATION_PERSONNEL', 'TS_MAJ_PPERSONNEL']
['ID_MISSION', 'ID_PERSONNEL', 'NOM_PERSONNEL', 'PRENOM_PERSONNEL', 'DATE_MISSION', 'TYPE_MISSION', 'VILLE_DEPART', 'PAYS_DEPART', 'VILLE_DESTINATION', 'PAYS_DESTINATION', 'TRANSPORT', 'ALLER_RETOUR']


# Tables
- FAITS MISSION
    - ID_PERSONNEL
    - ID_MISSION
    - ANNEE
    - MOIS
    - JOUR
    - SITE
- FAITS MATERIEL INFORMATIQUE
    - ID_PERSONNEL
    - ID_MATERIELINFO
    - ANNEE
    - MOIS
    - JOUR
    - SITE

- DATE : ANNEE, MOIS, JOUR
    - (From DATE_MISSION/DATE_ACHAT)
        - ANNEE
        - MOIS
        - JOUR
- SITE : SITE
    - SITE (From PERSONNEL.VILLE)
- MISSION : ID_MISSION
    - (from MISSION) : ID_MISSION, VILLE_DEPART, VILLE_DESTINATION,TYPE_MISSION,TRANSPORT,ALLER_RETOUR
    - (From colonne calculée) : IMPACT CARBONE
- LOCALISATION : Ville
    - (From MISSION.VILLE_DEPART, MISSION.VILLE_DESTINATION) : VILLE
    - (From MISSION.PAYS_DEPART,MISSION.PAYS_DESTINATION) : PAYS
    - (From : colonne calculée (geopy ?)) : CONTINENT
- Personnel - ID_PERSONNEL
    - ID_PERSONNEL, PAYS, FONCTION_PERSONNEL, DT_NAISSANCE
- MATERIEL INFORMATIQUE - ID_MATERIELINFO
    - Matériel informatique : ID_MATERIELINFO, DATE_ACHAT, TYPE (Split et calculé), Ecran (calculé), MODELE
    - Materiel informatique impact : Impact (join on type and modèle)

In [5]:
import glob
fichiers_mat_inf = glob.glob('BDD_BGES/**/MATERIEL_INFORMATIQUE_*.txt', recursive=True)
psdf_mat_inf_list = [ps.read_csv(f, sep=';') for f in fichiers_mat_inf]
psdf_mat_inf = ps.concat(psdf_mat_inf_list, ignore_index=True)

fichiers_mission = glob.glob('BDD_BGES/**/MISSION_*.txt', recursive=True)
psdf_mission_list = [ps.read_csv(f, sep=';',index_col='ID_MISSION') for f in fichiers_mission]
psdf_mission = ps.concat(psdf_mission_list, ignore_index=False)


C:\Users\vdugr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyspark\pandas\utils.py:1038: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [6]:
from pyspark.sql.functions import *
sdf_mat_inf = psdf_mat_inf.to_spark()
sdf_mission = psdf_mission.to_spark(index_col='ID_MISSION')

DATE = sdf_mission.select(col('DATE_MISSION').alias('DATE')).join(sdf_mat_inf.select(col('DATE_ACHAT').alias('DATE')), on='DATE', how='outer')
DATE = DATE.select(to_date(col('DATE'), 'yyyy-MM-dd').alias('DATE')).distinct()
date_min_max = DATE.agg(
    min("DATE").alias("DATE_MIN"),
    max("DATE").alias("DATE_MAX")
)
date_min_max.show()

C:\Users\vdugr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyspark\pandas\utils.py:1038: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


+----------+----------+
|  DATE_MIN|  DATE_MAX|
+----------+----------+
|2026-04-29|2026-11-14|
+----------+----------+



In [7]:
import glob
#Charger les fichiers de personnel
fichiers_personnel = glob.glob('BDD_BGES/**/PERSONNEL_*.txt', recursive=True)
psdf_personnel_list = [ps.read_csv(f, sep=';',index_col='ID_PERSONNEL') for f in fichiers_personnel]
psdf_personnel = ps.concat(psdf_personnel_list, ignore_index=False)

#Charger les fichiers de matériel informatique
fichiers_mat_inf = glob.glob('BDD_BGES/**/MATERIEL_INFORMATIQUE_*.txt', recursive=True)
psdf_mat_inf_list = [ps.read_csv(f, sep=';') for f in fichiers_mat_inf]
psdf_mat_inf_temp = ps.concat(psdf_mat_inf_list, ignore_index=True)

#Charge les données d'impact carbone du matériel informatique
psdf_impact_matinf = ps.read_csv('BDD_BGES/materiel_informatique_impact.csv',sep=',')
psdf_impact_matinf = psdf_impact_matinf.rename(columns={
    'Type': 'TYPE',
    'Impact': 'IMPACT',
    'Modèle': 'MODELE'
})

#Fusionne les données de matériel informatique avec les données d'impact carbone
psdf_mat_inf = psdf_mat_inf_temp.merge(
    psdf_impact_matinf,
    on=['TYPE','MODELE'],
    how='inner'
)
#Impact est en kg/CO2, il faut le convertir en tonnes de CO2
psdf_mat_inf['IMPACT'] = psdf_mat_inf['IMPACT'] / 1000
psdf_mat_inf = psdf_mat_inf.set_index('ID_MATERIELINFO')

#Charger les fichiers de mission
fichiers_mission = glob.glob('BDD_BGES/**/MISSION_*.txt', recursive=True)
psdf_mission_list = [ps.read_csv(f, sep=';',index_col='ID_MISSION') for f in fichiers_mission]
psdf_mission = ps.concat(psdf_mission_list, ignore_index=False)

In [8]:
# Transformation en spark dataframe
from pyspark.sql.functions import *
sdf_personnel = psdf_personnel.to_spark(index_col='ID_PERSONNEL')
sdf_personnel.show(10)
sdf_mat_inf = psdf_mat_inf.to_spark(index_col='ID_MATERIELINFO')
sdf_mat_inf.show(10)
sdf_mission = psdf_mission.to_spark(index_col='ID_MISSION')
sdf_mission.show(10)

+--------------------+-------------+----------------+----------+------------+----------+-----------+-----------------+-------------+--------+----------+---------+---------+------+-------+------------------+---------------------+-------------------+
|        ID_PERSONNEL|NOM_PERSONNEL|PRENOM_PERSONNEL|  DT_NAISS| VILLE_NAISS|PAYS_NAISS|   NUM_SECU|IND_PAYS_NUM_TELP|NUM_TELEPHONE|NUM_VOIE|  DSC_VOIE|CMPL_VOIE|CD_POSTAL| VILLE|   PAYS|FONCTION_PERSONNEL|TS_CREATION_PERSONNEL|  TS_MAJ_PPERSONNEL|
+--------------------+-------------+----------------+----------+------------+----------+-----------+-----------------+-------------+--------+----------+---------+---------+------+-------+------------------+---------------------+-------------------+
|KeyPers_Berlin_12...|        Name0|       FistName0|1948-11-17|      Bogota|  Colombia|NS000000000|             NULL|+336##0530601|      51|NomVoie520|     NULL|    #8356|Berlin|Germany|    Dateningenieur|  2000-05-09 11:54:25|2000-05-09 11:54:25|
|Key

In [9]:
DATE = sdf_mission.select(col('DATE_MISSION').alias('DATE')).join(sdf_mat_inf.select(col('DATE_ACHAT').alias('DATE')), on='DATE', how='outer')
DATE = DATE.select(to_date(col('DATE'), 'yyyy-MM-dd').alias('DATE'))
DATE = DATE.select(year(col('DATE')).alias('ANNEE'), month(col('DATE')).alias('MOIS'), dayofmonth(col('DATE')).alias('JOUR'))
DATE.show(10)

+-----+----+----+
|ANNEE|MOIS|JOUR|
+-----+----+----+
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
+-----+----+----+
only showing top 10 rows


In [10]:
#Création du modèle étoile
FAITS_MISSION = sdf_mission.join(sdf_personnel, on='ID_PERSONNEL', how='inner').select('ID_MISSION','ID_PERSONNEL',col('VILLE').alias('SITE'),year(to_date(col('DATE_MISSION'))).alias('ANNEE'), month(to_date(col('DATE_MISSION'))).alias('MOIS'), dayofmonth(to_date(col('DATE_MISSION'))).alias('JOUR'))
FAITS_MISSION.show(10)

SITE = sdf_personnel.select(col('VILLE').alias('SITE')).distinct()
SITE.show(10)

+----------------+--------------------+------+-----+----+----+
|      ID_MISSION|        ID_PERSONNEL|  SITE|ANNEE|MOIS|JOUR|
+----------------+--------------------+------+-----+----+----+
|BERLIN_202604290|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604291|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604292|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604293|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604294|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604295|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604296|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604297|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604298|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604299|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
+----------------+--------------------+------+-----+----+----+
only showing top 10 rows
+-----------+
|       SITE|
+-----------+
|     Berlin|
|     London|
|Los Angeles|
|   New-Y

In [15]:
import pandas as pd
import time
from geopy.geocoders import Nominatim

from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

geolocator = Nominatim(user_agent="geo_app") #init du géolocaliseur

#Récupérer toutes les villes uniques
dep = sdf_mission.select(
    col("VILLE_DEPART").alias("VILLE"),
    col("PAYS_DEPART").alias("PAYS")
)

arr = sdf_mission.select(
    col("VILLE_DESTINATION").alias("VILLE"),
    col("PAYS_DESTINATION").alias("PAYS")
)

villes = dep.union(arr).distinct().toPandas()


#Géocoder les villes et stocker les résultats dans un cache local pour éviter de refaire les requêtes à chaque exécution
coords = []
if os.path.exists('CACHE_COORD.csv'):
    coord_csv = ps.read_csv('CACHE_COORD.csv',sep=',',index_col='VILLE')
    villes_existantes = set(coord_csv['VILLE'])
else :
    coord_csv = None
    villes_existantes = set()

for _, row in villes.iterrows():
    ville = row["VILLE"]
    pays = row["PAYS"]

    if pd.notna(ville) and pd.notna(pays):
        if ville in villes_existantes:
            query = f"{ville}, {pays}"
            try:
                loc = geolocator.geocode(query, timeout=10)
                if loc:
                    coords.append((ville, pays, float(loc.latitude), float(loc.longitude)))
                else:
                    print(f"Non trouvé : {query}")
                time.sleep(1)
            except Exception as e:
                print(f"Erreur géocodage {query}: {e}")


schema_coords = StructType([
    StructField("VILLE", StringType(), True),
    StructField("PAYS", StringType(), True),
    StructField("LAT", DoubleType(), True),
    StructField("LON", DoubleType(), True)
])

if coords != []:
    coords_pdf = pd.DataFrame(coords, columns=["VILLE", "PAYS", "LAT", "LON"])
    coords_pdf.to_csv("CACHE_COORD.csv", index=False, encoding="utf-8", mode='a',header=not os.path.exists('CACHE_COORD.csv'))

# Charger les coordonnées depuis le cache
sdf_coords = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("CACHE_COORD.csv")

sdf_coords = sdf_coords \
    .withColumn("LAT", col("LAT").cast("double")) \
    .withColumn("LON", col("LON").cast("double"))


# Joindre les coordonnées départ / arrivée
coords_dep = sdf_coords.select(
    col("VILLE").alias("VILLE_DEPART"),
    col("PAYS").alias("PAYS_DEPART"),
    col("LAT").alias("LAT_DEP"),
    col("LON").alias("LON_DEP")
)

coords_arr = sdf_coords.select(
    col("VILLE").alias("VILLE_DESTINATION"),
    col("PAYS").alias("PAYS_DESTINATION"),
    col("LAT").alias("LAT_ARR"),
    col("LON").alias("LON_ARR")
)

MISSION = sdf_mission \
    .join(coords_dep, on=["VILLE_DEPART", "PAYS_DEPART"], how="left") \
    .join(coords_arr, on=["VILLE_DESTINATION", "PAYS_DESTINATION"], how="left")

# Calcul distance haversine en Spark
R = 6371.0 # Rayon de la Terre en km

lat1 = radians(col("LAT_DEP"))
lon1 = radians(col("LON_DEP"))
lat2 = radians(col("LAT_ARR"))
lon2 = radians(col("LON_ARR"))

dlat = lat2 - lat1
dlon = lon2 - lon1

a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
distance_expr = 2 * lit(R) * asin(sqrt(a))

# Ajouter la distance au dataframe de mission
MISSION = MISSION.withColumn(
    "DISTANCE_KM",
    when(
        col("LAT_DEP").isNotNull() & col("LAT_ARR").isNotNull(),
        distance_expr
    ).otherwise(
        when(
            col("VILLE_DEPART") == col("VILLE_DESTINATION"),
            lit(4.0) #Selon une étude sur la distance des trajets urbains https://www.mdpi.com/2072-4292/13/9/1825
        ).otherwise(None) #Si on n'a pas les coordonnées et que les villes sont différentes, on ne peut pas calculer la distance
    )
)

# 5) Calcul impact carbone
mode = lower(trim(col("TRANSPORT")))

MISSION = MISSION.withColumn(
    "IMPACT_CARBONE",
    when(
        col("DISTANCE_KM").isNotNull(),
        when(mode == "avion",
            when(col("DISTANCE_KM") < 1000, col("DISTANCE_KM") * 0.000258) # Coefficient pour les vols courts
            .when(col("DISTANCE_KM") < 3500, col("DISTANCE_KM") * 0.000187) # Coefficient pour les vols moyens
            .otherwise(col("DISTANCE_KM") * 0.000152) # Coefficient pour les vols longs
        )
        .when(mode == "train", col("DISTANCE_KM") * 0.000004) # Coefficient pour le train
        .when(mode == "taxi", col("DISTANCE_KM") * 0.000192) # Coefficient pour le taxi
        .when(mode == "transports en commun", col("DISTANCE_KM") * 0.00005) # Coefficient pour les transports en commun
        .otherwise(None) # Si le mode de transport n'est pas reconnu, on ne peut pas calculer l'impact carbone
    ).otherwise(None)
)

# Vérification
MISSION.select(
    "VILLE_DEPART",
    "PAYS_DEPART",
    "VILLE_DESTINATION",
    "PAYS_DESTINATION",
    "TRANSPORT",
    "DISTANCE_KM",
    "IMPACT_CARBONE"
).show(20, truncate=False)
#Géocoder les villes et stocker les résultats dans un cache local pour éviter de refaire les requêtes à chaque exécution
coords = []
if os.path.exists('CACHE_COORD.csv'):
    coord_csv = ps.read_csv('CACHE_COORD.csv',sep=',',index_col='VILLE')
    villes_existantes = set(coord_csv.index.to_list())
else :
    coord_csv = None
    villes_existantes = set()
spark_coord_csv = coord_csv.to_spark(index_col='VILLE')

for _, row in villes.iterrows():
    ville = row["VILLE"]
    pays = row["PAYS"]

    if pd.notna(ville) and pd.notna(pays):
        if ville in villes_existantes:
            query = f"{ville}, {pays}"
            try:
                loc = geolocator.geocode(query, timeout=10)
                if loc:
                    coords.append((ville, pays, float(loc.latitude), float(loc.longitude)))
                else:
                    print(f"Non trouvé : {query}")
                time.sleep(1)
            except Exception as e:
                print(f"Erreur géocodage {query}: {e}")


schema_coords = StructType([
    StructField("VILLE", StringType(), True),
    StructField("PAYS", StringType(), True),
    StructField("LAT", DoubleType(), True),
    StructField("LON", DoubleType(), True)
])

if coords != []:
    coords_pdf = pd.DataFrame(coords, columns=["VILLE", "PAYS", "LAT", "LON"])
    coords_pdf.to_csv("CACHE_COORD.csv", index=False, encoding="utf-8", mode='a',header=not os.path.exists('CACHE_COORD.csv'))

# Charger les coordonnées depuis le cache
sdf_coords = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("CACHE_COORD.csv")

sdf_coords = sdf_coords \
    .withColumn("LAT", col("LAT").cast("double")) \
    .withColumn("LON", col("LON").cast("double"))


# Joindre les coordonnées départ / arrivée
coords_dep = sdf_coords.select(
    col("VILLE").alias("VILLE_DEPART"),
    col("PAYS").alias("PAYS_DEPART"),
    col("LAT").alias("LAT_DEP"),
    col("LON").alias("LON_DEP")
)

coords_arr = sdf_coords.select(
    col("VILLE").alias("VILLE_DESTINATION"),
    col("PAYS").alias("PAYS_DESTINATION"),
    col("LAT").alias("LAT_ARR"),
    col("LON").alias("LON_ARR")
)

MISSION = sdf_mission \
    .join(coords_dep, on=["VILLE_DEPART", "PAYS_DEPART"], how="left") \
    .join(coords_arr, on=["VILLE_DESTINATION", "PAYS_DESTINATION"], how="left")

# Calcul distance haversine en Spark
R = 6371.0 # Rayon de la Terre en km

lat1 = radians(col("LAT_DEP"))
lon1 = radians(col("LON_DEP"))
lat2 = radians(col("LAT_ARR"))
lon2 = radians(col("LON_ARR"))

dlat = lat2 - lat1
dlon = lon2 - lon1

a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
distance_expr = 2 * lit(R) * asin(sqrt(a))

# Ajouter la distance au dataframe de mission
MISSION = MISSION.withColumn(
    "DISTANCE_KM",
    when(
        col("LAT_DEP").isNotNull() & col("LAT_ARR").isNotNull(),
        distance_expr
    ).otherwise(
        when(
            col("VILLE_DEPART") == col("VILLE_DESTINATION"),
            lit(4.0) #Selon une étude sur la distance des trajets urbains https://www.mdpi.com/2072-4292/13/9/1825
        ).otherwise(None) #Si on n'a pas les coordonnées et que les villes sont différentes, on ne peut pas calculer la distance
    )
)

# Calcul impact carbone en Spark
mode = lower(trim(col("TRANSPORT")))

MISSION = MISSION.withColumn(
    "IMPACT_CARBONE",
    when(
        col("DISTANCE_KM").isNotNull(),
        when(mode == "avion",
            when(col("DISTANCE_KM") < 1000, col("DISTANCE_KM") * 0.000258) # Coefficient pour les vols courts
            .when(col("DISTANCE_KM") < 3500, col("DISTANCE_KM") * 0.000187) # Coefficient pour les vols moyens
            .otherwise(col("DISTANCE_KM") * 0.000152) # Coefficient pour les vols longs
        )
        .when(mode == "train", col("DISTANCE_KM") * 0.000004) # Coefficient pour le train
        .when(mode == "taxi", col("DISTANCE_KM") * 0.000192) # Coefficient pour le taxi
        .when(mode == "transports en commun", col("DISTANCE_KM") * 0.00005) # Coefficient pour les transports en commun
        .otherwise(None) # Si le mode de transport n'est pas reconnu, on ne peut pas calculer l'impact carbone
    ).otherwise(None)
)


MISSION  = MISSION.select(
    "ID_MISSION",
    "TYPE_MISSION",
    "VILLE_DEPART",
    "VILLE_DESTINATION",
    "TRANSPORT",
    "ALLER_RETOUR",
    "IMPACT_CARBONE"
)

KeyError: 'VILLE'

In [ ]:
#Géocoder les villes et stocker les résultats dans un cache local pour éviter de refaire les requêtes à chaque exécution
coords = []
if os.path.exists('CACHE_COORD.csv'):
    coord_csv = ps.read_csv('CACHE_COORD.csv',sep=',',index_col='VILLE')
    villes_existantes = set(coord_csv.index.to_list())
else :
    coord_csv = None
    villes_existantes = set()
spark_coord_csv = coord_csv.to_spark(index_col='VILLE')

for _, row in villes.iterrows():
    ville = row["VILLE"]
    pays = row["PAYS"]

    if pd.notna(ville) and pd.notna(pays):
        if ville in villes_existantes:
            query = f"{ville}, {pays}"
            try:
                loc = geolocator.geocode(query, timeout=10)
                if loc:
                    coords.append((ville, pays, float(loc.latitude), float(loc.longitude)))
                else:
                    print(f"Non trouvé : {query}")
                time.sleep(1)
            except Exception as e:
                print(f"Erreur géocodage {query}: {e}")


schema_coords = StructType([
    StructField("VILLE", StringType(), True),
    StructField("PAYS", StringType(), True),
    StructField("LAT", DoubleType(), True),
    StructField("LON", DoubleType(), True)
])

if coords != []:
    coords_pdf = pd.DataFrame(coords, columns=["VILLE", "PAYS", "LAT", "LON"])
    coords_pdf.to_csv("CACHE_COORD.csv", index=False, encoding="utf-8", mode='a',header=not os.path.exists('CACHE_COORD.csv'))

# Charger les coordonnées depuis le cache
sdf_coords = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("CACHE_COORD.csv")

sdf_coords = sdf_coords \
    .withColumn("LAT", col("LAT").cast("double")) \
    .withColumn("LON", col("LON").cast("double"))


# Joindre les coordonnées départ / arrivée
coords_dep = sdf_coords.select(
    col("VILLE").alias("VILLE_DEPART"),
    col("PAYS").alias("PAYS_DEPART"),
    col("LAT").alias("LAT_DEP"),
    col("LON").alias("LON_DEP")
)

coords_arr = sdf_coords.select(
    col("VILLE").alias("VILLE_DESTINATION"),
    col("PAYS").alias("PAYS_DESTINATION"),
    col("LAT").alias("LAT_ARR"),
    col("LON").alias("LON_ARR")
)

MISSION = sdf_mission \
    .join(coords_dep, on=["VILLE_DEPART", "PAYS_DEPART"], how="left") \
    .join(coords_arr, on=["VILLE_DESTINATION", "PAYS_DESTINATION"], how="left")

# Calcul distance haversine en Spark
R = 6371.0 # Rayon de la Terre en km

lat1 = radians(col("LAT_DEP"))
lon1 = radians(col("LON_DEP"))
lat2 = radians(col("LAT_ARR"))
lon2 = radians(col("LON_ARR"))

dlat = lat2 - lat1
dlon = lon2 - lon1

a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
distance_expr = 2 * lit(R) * asin(sqrt(a))

# Ajouter la distance au dataframe de mission
MISSION = MISSION.withColumn(
    "DISTANCE_KM",
    when(
        col("LAT_DEP").isNotNull() & col("LAT_ARR").isNotNull(),
        distance_expr
    ).otherwise(
        when(
            col("VILLE_DEPART") == col("VILLE_DESTINATION"),
            lit(4.0) #Selon une étude sur la distance des trajets urbains https://www.mdpi.com/2072-4292/13/9/1825
        ).otherwise(None) #Si on n'a pas les coordonnées et que les villes sont différentes, on ne peut pas calculer la distance
    )
)

# Calcul impact carbone en Spark
mode = lower(trim(col("TRANSPORT")))

MISSION = MISSION.withColumn(
    "IMPACT_CARBONE",
    when(
        col("DISTANCE_KM").isNotNull(),
        when(mode == "avion",
            when(col("DISTANCE_KM") < 1000, col("DISTANCE_KM") * 0.000258) # Coefficient pour les vols courts
            .when(col("DISTANCE_KM") < 3500, col("DISTANCE_KM") * 0.000187) # Coefficient pour les vols moyens
            .otherwise(col("DISTANCE_KM") * 0.000152) # Coefficient pour les vols longs
        )
        .when(mode == "train", col("DISTANCE_KM") * 0.000004) # Coefficient pour le train
        .when(mode == "taxi", col("DISTANCE_KM") * 0.000192) # Coefficient pour le taxi
        .when(mode == "transports en commun", col("DISTANCE_KM") * 0.00005) # Coefficient pour les transports en commun
        .otherwise(None) # Si le mode de transport n'est pas reconnu, on ne peut pas calculer l'impact carbone
    ).otherwise(None)
)


MISSION  = MISSION.select(
    "ID_MISSION",
    "TYPE_MISSION",
    "VILLE_DEPART",
    "VILLE_DESTINATION",
    "TRANSPORT",
    "ALLER_RETOUR",
    "IMPACT_CARBONE"
)

c:\Users\rzong\anaconda3\envs\nf26_env\Lib\site-packages\pyspark\pandas\utils.py:1038: PandasAPIOnSparkAdviceWarning: `to_list` loads all data into the driver's memory. It should only be used if the resulting list is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


+------------+-----------+-----------------+----------------+---------+------------------+------------------+
|VILLE_DEPART|PAYS_DEPART|VILLE_DESTINATION|PAYS_DESTINATION|TRANSPORT|DISTANCE_KM       |IMPACT_CARBONE    |
+------------+-----------+-----------------+----------------+---------+------------------+------------------+
|Berlin      |Allemagne  |Washington       |USA             |Avion    |6710.669579367729 |1.0200217760638948|
|Berlin      |Allemagne  |Washington       |USA             |Avion    |6710.669579367729 |1.0200217760638948|
|Berlin      |Allemagne  |Washington       |USA             |Avion    |6710.669579367729 |1.0200217760638948|
|Berlin      |Allemagne  |Washington       |USA             |Avion    |6710.669579367729 |1.0200217760638948|
|Berlin      |Allemagne  |Alger            |Algeria         |Avion    |1927.7293922968229|0.3604853963595059|
|Berlin      |Allemagne  |Alger            |Algeria         |Avion    |1927.7293922968229|0.3604853963595059|
|Berlin   

In [ ]:
MISSION  = MISSION.select(
    "ID_MISSION",
    "TYPE_MISSION",
    "VILLE_DEPART",
    "VILLE_DESTINATION",
    "TRANSPORT",
    "ALLER_RETOUR",
    "IMPACT_CARBONE"
)
MISSION.show(20)

+----------------+------------+------------+-----------------+---------+------------+------------------+
|      ID_MISSION|TYPE_MISSION|VILLE_DEPART|VILLE_DESTINATION|TRANSPORT|ALLER_RETOUR|    IMPACT_CARBONE|
+----------------+------------+------------+-----------------+---------+------------+------------------+
|BERLIN_202604290|     Meeting|      Berlin|       Washington|    Avion|         oui|1.0200217760638948|
|BERLIN_202604290|     Meeting|      Berlin|       Washington|    Avion|         oui|1.0200217760638948|
|BERLIN_202604290|     Meeting|      Berlin|       Washington|    Avion|         oui|1.0200217760638948|
|BERLIN_202604290|     Meeting|      Berlin|       Washington|    Avion|         oui|1.0200217760638948|
|BERLIN_202604291|    Schulung|      Berlin|            Alger|    Avion|         oui|0.3604853963595059|
|BERLIN_202604291|    Schulung|      Berlin|            Alger|    Avion|         oui|0.3604853963595059|
|BERLIN_202604291|    Schulung|      Berlin|           

In [ ]:
import pyspark.pandas as ps
co = ps.read_csv('coords_temp.csv',sep=',',index_col='VILLE')
co_s = co.to_spark(index_col='VILLE')

In [ ]:
co_s[co_s['VILLE']=='Paris'].collect()==[]

False

In [ ]:
#Création de la table LOCALISATION
from pyspark.sql.types import StringType
import country_converter as coco
from pyspark.sql.functions import udf
from deep_translator import GoogleTranslator

cc = coco.CountryConverter()

def get_continent(pays):
    try:
        continent = cc.convert(names=pays, to='continent')
        if continent == "not found":
            return None
        return continent
    except:
        return None
    

LOCALISATION = sdf_mission.select(
    col("VILLE_DEPART").alias("VILLE"),
    col("PAYS_DEPART").alias("PAYS")
).union(
    sdf_mission.select(
        col("VILLE_DESTINATION").alias("VILLE"),
        col("PAYS_DESTINATION").alias("PAYS")
    )
).distinct()

rows = LOCALISATION.collect()
pays_uniques = [row["PAYS"] for row in LOCALISATION.select("PAYS").distinct().collect()]
translator = GoogleTranslator(source='auto', target='en')

dict_pays_en = {
    pays: ("Sweden" if pays.lower() in ["suede", "suède"] else translator.translate(pays))
    for pays in pays_uniques
}

dict_pays_continent = {
    pays_fr: get_continent(dict_pays_en[pays_fr])
    for pays_fr in dict_pays_en
}

mapping_list = []
for k, v in dict_pays_continent.items():
    mapping_list += [lit(k), lit(v)]

mapping_expr = create_map(*mapping_list)
LOCALISATION = LOCALISATION.withColumn(
    "CONTINENT",
    mapping_expr[col("PAYS")]
)
LOCALISATION.show(10)

+-----------+-----------+---------+
|      VILLE|       PAYS|CONTINENT|
+-----------+-----------+---------+
|   Shanghai|      China|     Asia|
|     Berlin|  Allemagne|   Europe|
|      Paris|     France|   Europe|
|   New-York|        USA|  America|
|Los Angeles|        USA|  America|
|     London|    England|   Europe|
|      Dubaï|    Emirats|     Asia|
| Wellington|New Zealand|  Oceania|
|   Helsinki|   Finlande|   Europe|
|  Vancouver|     Canada|  America|
+-----------+-----------+---------+
only showing top 10 rows


In [ ]:
FAITS_MATERIEL_INFORMATIQUE = sdf_mat_inf.join(sdf_personnel, on='ID_PERSONNEL', how='inner').select('ID_PERSONNEL', 'ID_MATERIELINFO', year(to_date(col('DATE_ACHAT'))).alias('ANNEE'), month(to_date(col('DATE_ACHAT'))).alias('MOIS'), dayofmonth(to_date(col('DATE_ACHAT'))).alias('JOUR'), col('VILLE').alias('SITE'))
FAITS_MATERIEL_INFORMATIQUE.show(10)

+--------------------+--------------------+-----+----+----+------+
|        ID_PERSONNEL|     ID_MATERIELINFO|ANNEE|MOIS|JOUR|  SITE|
+--------------------+--------------------+-----+----+----+------+
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   4|  29|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   4|  29|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   4|  29|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   4|  30|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   4|  30|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   4|  30|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   5|   1|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   5|   2|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   5|   2|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   5|   2|Berlin|
+--------------------+--------------------+-----+----+----+------+
only showing top 10 rows


In [ ]:
PERSONNEL = sdf_personnel.select('ID_PERSONNEL','PAYS','FONCTION_PERSONNEL',col('DT_NAISS').alias('DATE_NAISSANCE'))
PERSONNEL.show(10)

+--------------------+-------+------------------+--------------+
|        ID_PERSONNEL|   PAYS|FONCTION_PERSONNEL|DATE_NAISSANCE|
+--------------------+-------+------------------+--------------+
|KeyPers_Berlin_12...|Germany|    Dateningenieur|    1948-11-17|
|KeyPers_Berlin_12...|Germany|     Führungskraft|    1941-03-05|
|KeyPers_Berlin_12...|Germany| Computeringenieur|    1958-09-14|
|KeyPers_Berlin_12...|Germany| Computeringenieur|    1961-04-18|
|KeyPers_Berlin_12...|Germany|    Dateningenieur|    1950-09-08|
|KeyPers_Berlin_12...|Germany| Computeringenieur|    1977-09-09|
|KeyPers_Berlin_12...|Germany| Computeringenieur|    2000-10-08|
|KeyPers_Berlin_12...|Germany|     Führungskraft|    1997-04-08|
|KeyPers_Berlin_12...|Germany|    Dateningenieur|    1933-04-07|
|KeyPers_Berlin_12...|Germany| Computeringenieur|    1975-03-12|
+--------------------+-------+------------------+--------------+
only showing top 10 rows


In [ ]:
from pyspark.sql.functions import col, when, lower, trim, split, count, desc, regexp_replace, lit, coalesce
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

MATERIEL_INFORMATIQUE = sdf_mat_inf.select(
    "ID_MATERIELINFO", "TYPE", "MODELE", "IMPACT"
)

# 1) Nettoyage de base
MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE \
    .withColumn("TYPE", lower(trim(col("TYPE")))) \
    .withColumn("MODELE", lower(trim(col("MODELE")))) \
    .withColumn(
        "TYPE",
        when((col("TYPE").isNull()) | (col("TYPE") == "") | (col("TYPE") == "nan"), None)
        .otherwise(col("TYPE"))
    ) \
    .withColumn(
        "MODELE",
        when((col("MODELE").isNull()) | (col("MODELE") == "") | (col("MODELE") == "nan"), None)
        .otherwise(col("MODELE"))
    )

# 2) Prédiction TYPE par MODELE : type majoritaire pour chaque modèle
modele_type_count = MATERIEL_INFORMATIQUE \
    .filter(col("TYPE").isNotNull() & col("MODELE").isNotNull()) \
    .groupBy("MODELE", "TYPE") \
    .agg(count("*").alias("nb"))

window_modele = Window.partitionBy("MODELE").orderBy(desc("nb"))

modele_type_pred = modele_type_count \
    .withColumn("rank", row_number().over(window_modele)) \
    .filter(col("rank") == 1) \
    .select("MODELE", col("TYPE").alias("TYPE_PRED_MODELE"))

# 3) Fallback global : type le plus fréquent du dataset
global_type_pred = MATERIEL_INFORMATIQUE \
    .filter(col("TYPE").isNotNull()) \
    .groupBy("TYPE") \
    .agg(count("*").alias("nb")) \
    .orderBy(desc("nb")) \
    .limit(1) \
    .collect()[0]["TYPE"]

# 4) Imputation TYPE
MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE \
    .join(modele_type_pred, on="MODELE", how="left") \
    .withColumn(
        "TYPE",
        coalesce(col("TYPE"), col("TYPE_PRED_MODELE"), lit(global_type_pred))
    ) \
    .drop("TYPE_PRED_MODELE")

# 5) Détection écran AVANT nettoyage final
MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE.withColumn(
    "ECRAN",
    when(col("TYPE").contains("sans ecran"), "non")
    .when(col("TYPE").contains("sans écran"), "non")
    .otherwise("oui")
)

# 6) Nettoyage TYPE final
MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE.withColumn(
    "TYPE",
    regexp_replace(col("TYPE"), "sans ecran|sans écran", "")
)

MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE.withColumn(
    "TYPE",
    trim(col("TYPE"))
)

# 7) Résultat final
MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE.select(
    "ID_MATERIELINFO", "TYPE", "MODELE", "IMPACT", "ECRAN"
)

MATERIEL_INFORMATIQUE.show(10, truncate=False)

+--------------------+------------------+--------------------+------+-----+
|     ID_MATERIELINFO|              TYPE|              MODELE|IMPACT|ECRAN|
+--------------------+------------------+--------------------+------+-----+
|BERLIN_MATERIEL_I...|       pc portable|       EliteBook 6xx| 0.175|  oui|
|BERLIN_MATERIEL_I...|  vidéo projecteur|   modèle par défaut| 0.094|  oui|
|BERLIN_MATERIEL_I...|           pc fixe|Precision tower 7xxx| 0.555|  non|
|BERLIN_MATERIEL_I...|           pc fixe|EliteDesk (Tower,...| 0.393|  non|
|BERLIN_MATERIEL_I...|       pc portable|      EliteBook 10xx| 0.223|  oui|
|BERLIN_MATERIEL_I...|       pc portable|         MacBook air|  0.25|  oui|
|BERLIN_MATERIEL_I...|             ecran|    32pouces et plus|  0.59|  oui|
|BERLIN_MATERIEL_I...|pc fixe tout-en-un|Optiplex All-in-O...| 0.318|  oui|
|BERLIN_MATERIEL_I...|           pc fixe|Precision tower 3xxx| 0.285|  non|
|BERLIN_MATERIEL_I...|             ecran|   24pouces-31pouces|  0.43|  oui|
+-----------

{'Geschäftstreffen': 'Business Meeting',
 'Konferenz': 'Conference',
 'Schulung': 'Training',
 'Meeting': 'Internal Meeting',
 'Entwicklung': 'Development',
 'Business Meeting': 'Business Meeting',
 'Conference': 'Conference',
 'Team Meeting': 'Internal Meeting',
 'Development': 'Development',
 'Vocational Training': 'Training',
 'Conférence': 'Conference',
 'Formation': 'Training',
 'Réunion': 'Internal Meeting',
 'Rencontre entreprises': 'Business Meeting',
 'Développement': 'Development'}

In [ ]:
from pyspark.sql.functions import col, count, when, current_date, trim

# Fonction pour compter les valeurs nulles
def count_nulls(df, name):
    print(f"Null values dans {name}:")
    df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show(truncate=False)
    print("\n")

# Helper pour détecter les chaînes vides ou nulles
def is_blank(column):
    return column.isNull() | (trim(column) == "")

# Fonction pour afficher les lignes suspectes selon un filtre
def show_suspect_rows(df, name, condition, message):
    print(f"{name} - {message}")
    df.filter(condition).show(20, truncate=False)
    print("\n")

# Comptage global des valeurs manquantes
count_nulls(Faits, "Faits")
count_nulls(Trajets, "Trajets")
count_nulls(Personnel, "Personnel")
count_nulls(MaterielInformatique, "MaterielInformatique")

# Recherche des valeurs aberrantes ou incohérences dans les tables dérivées
show_suspect_rows(
    Faits,
    "Faits",
    col("ID_MISSION").isNull() | col("ID_PERSONNEL").isNull() | col("ID_MATERIELINFO").isNull(),
    "lignes avec clés manquantes"
)

show_suspect_rows(
    Trajets,
    "Trajets",
    is_blank(col("PAYS_DEPART")) |
    is_blank(col("PAYS_DESTINATION")) |
    is_blank(col("VILLE_DEPART")) |
    is_blank(col("VILLE_DESTINATION")),
    "trajets avec pays ou villes manquants"
)

show_suspect_rows(
    Trajets,
    "Trajets",
    col("VILLE_DEPART") == col("VILLE_DESTINATION"),
    "trajets où la ville de départ est identique à la ville de destination"
)

show_suspect_rows(
    Personnel,
    "Personnel",
    is_blank(col("VILLE")) |
    is_blank(col("PAYS")) |
    is_blank(col("FONCTION_PERSONNEL")) |
    (col("TS_CREATION_PERSONNEL") > current_date()) |
    (col("TS_MAJ_PPERSONNEL") > current_date()),
    "personnel avec informations manquantes ou dates futures"
)

show_suspect_rows(
    MaterielInformatique,
    "MaterielInformatique",
    is_blank(col("TYPE")) |
    is_blank(col("MODELE")) |
    col("IMPACT").isNull() |
    (col("IMPACT") <= 0) |
    (col("DATE_ACHAT") > current_date()),
    "matériel avec type/modèle manquant, impact incorrect ou date d'achat future"
)

NameError: name 'Faits' is not defined

## ETL : Normalisation des données
#### A faire : 
- Tout traduire en anglais (exemple la fonction de directeur à berlin s'appelle Führungskraft)

## Requête sur le DataWarehouse

#### 1. Combien de cadres travaillent sur le site de Paris?

#### 2. Combien d’ingénieurs Data travaillent sur les sites aux États-Unis ?

#### 3. Combien d’ingénieurs informaticiens travaillent dans l’organisation (tous sites compris) ?

#### 4. Combien de PC fixes ont été achetés par l’organisation entre juin et septembre 2026 ?

#### 5. Quelle a été l’impact carbone des PC fixes sans ecran entre mai et octobre 2026 ?

#### 6. Quel a été l’impact carbone des PC portables achetés par les ingénieurs Data entre mai et octobre 2026 sur les sites de Londres et New-York?

#### 7. Quel a été l’impact carbone des écrans achetés par les cadres entre juillet et septembre 2026 sur tous les sites de l’organisation ?

#### 8. Quel a été l’impact carbone des missions sur les sites Européens entre mai et octobre 2026 ?

#### 9. Quels ont été les 5 jours les plus impactants concernant les missions en avion pour les sites Européens de l’organisation ?

#### 10. Quel a été le secteur d’activité qui a eu le plus d’impact concernant les missions et le matériel informatique sur l’ensemble des sites de l’organisation ?

#### 11.  Quel site a eu le plus d’impact concernant les missions et le matériel informatique sur l’ensemble des sites de l’organisation ?

#### 12. Quel a été l’impact carbone des missions reliant chaque site (la ville de départ est un site de l’organisation et la ville d’arrivée également) durant le mois de septembre 2026 ?

#### 13. Quel a été l’impact carbone des séminaires en juillet 2026 pour les employés de Los Angeles ?

#### 14. Quel secteur d’activité a été le plus impactant pour les missions “conférences” entre mai et septembre 2026 ?

#### 15. Quel a été l’âge moyen des employés Ingénieurs Data qui sont partis en formations entre juillet et septembre 2026 ?

#### 16. Quelle destination a été la plus impactante (en cumul) entre mai et octobre 2026 ?

#### 17. Quelles ont été les trois catégories de missions les plus impactantes pour les cadres dans les sites Européens en mai 2026 ?

*Les réponses des questions suivantes devront être sous forme de figures*
#### 18. Quelles ont été les 5 missions les plus impactantes sur le site de Paris ?

#### 19. Proposer une figure comparant l’impact carbone mensuel des missions en fonction du type de transport et sur chaque site.

#### 20. Proposer une figure illustrant l’impact carbone global mensuel de l’organisation.